# Volume 9, Chapter 5: Dedekind Domains (Python companion)

Ideal arithmetic and class numbers of quadratic fields, the fundamental identity checked on a cubic field, and the cyclotomic rings behind lattice-based cryptography with the number-theoretic transform.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★★) Ideal arithmetic in $\Z[\sqrt{-5}]$ and class numbers.** An ideal of $R=\Z[\sqrt{-5}]$ is a subgroup of $\Z^2$ (coordinates in the basis $1,\sqrt{-5}$), and can be stored as a $2\times2$ matrix in Hermite normal form; its norm $N(I)=|R/I|$ is the determinant. (a) Implement the product of ideals given by generators, and check the factorisations of Example 5.8: $2R=\mathfrak p^2$, $3R=\mathfrak q\bar{\mathfrak q}$, $(1\pm\sqrt{-5})=\mathfrak p\mathfrak q,\ \mathfrak p\bar{\mathfrak q}$. (b) An ideal $I$ is principal if and only if it contains an element of norm $N(I)$. Test $\mathfrak p$, $\mathfrak q$, $\mathfrak p\mathfrak q$ and $\mathfrak q^2$. (c) For a negative discriminant $D$, the class number $h(D)$ of the quadratic ring of discriminant $D$ is the number of *reduced* forms $ax^2+bxy+cy^2$ with $b^2-4ac=D$, $|b|\le a\le c$, $\gcd(a,b,c)=1$, and $b\ge0$ when $|b|=a$ or $a=c$ (Gauss; see Volume 12). Compute $h(-20)$, and list the fundamental discriminants $-2000<D<0$ with $h(D)=1$.

**P5.2 (★) The fundamental identity, checked.** The ring of integers of $\Q(\sqrt[3]2)$ is $\Z[\sqrt[3]2]$, so (Kummer–Dedekind, Volume 12) the factorisation of $p$ copies the factorisation of $x^3-2$ modulo $p$: a factor $g^e$ with $\deg g=f$ gives a prime with ramification index $e$ and residue degree $f$. For the primes below $200$, tabulate the pairs $(e_i,f_i)$ and check Theorem 5.14, $\sum e_if_i=3$. Which primes ramify, and which splitting types occur?

**P5.3 (★★) CS application: the rings of lattice cryptography.** The post-quantum schemes based on *Ring-LWE*, such as ML-KEM (Kyber, standardised by NIST in 2024 as FIPS 203), compute in $R_q=\Z_q[x]/(x^n+1)$ with $n$ a power of $2$. Here $\Z[x]/(x^n+1)$ is the ring of integers of the cyclotomic field $\Q(\zeta_{2n})$, a Dedekind domain, and $R_q=S/qS$ splits, by the Chinese remainder theorem, as in the proof of Theorem 5.14. (a) If $q\equiv1\pmod{2n}$, then $q$ splits completely: $x^n+1=\prod(x-\psi^{2i+1})$ modulo $q$ for a primitive $2n$-th root of unity $\psi$, and $R_q\cong\mathbb F_q^n$. Use this for $n=512$, $q=12289$ to multiply in $R_q$ with a fast *number-theoretic transform* (NTT) in $O(n\log n)$ operations, check the result against the schoolbook product, and time both. (b) Kyber uses $n=256$, $q=3329$. Show that $q\not\equiv1\pmod{512}$, so $x^{256}+1$ has no root modulo $q$, but that it splits into $128$ quadratic factors $x^2-\zeta^{2i+1}$ with $\zeta=17$ of order $256$; so $e=1$, $f=2$, $g=128$, and $efg=256$. (Volume C4 treats the cryptography.)

## Solutions

### P5.1

In [ ]:
from math import gcd, isqrt
from itertools import product

# Elements of Z[sqrt(-5)] are pairs (a, b) = a + b*sqrt(-5).
def mul(u, v):  return (u[0]*v[0] - 5*u[1]*v[1], u[0]*v[1] + u[1]*v[0])
def norm(u):    return u[0]**2 + 5*u[1]**2

def hnf(vectors):
    """Hermite normal form of the subgroup of Z^2 spanned by the vectors: rows (a, b), (0, d) with a, d > 0, 0 <= b < d."""
    vs = [list(v) for v in vectors if v != (0, 0)]
    # gcd on the first coordinate
    while sum(1 for v in vs if v[0] != 0) > 1:
        vs.sort(key=lambda v: (v[0] == 0, abs(v[0])))
        p = vs[0]
        for v in vs[1:]:
            if v[0] != 0:
                k = v[0] // p[0]; v[0] -= k*p[0]; v[1] -= k*p[1]
    vs.sort(key=lambda v: (v[0] == 0, abs(v[0])))
    top = vs[0] if vs[0][0] < 0 else vs[0]
    if top[0] < 0: top = [-top[0], -top[1]]
    d = 0
    for v in vs[1:]: d = gcd(d, v[1])
    b = top[1] % d if d else top[1]
    return ((top[0], b), (0, d))

def ideal(*gens):
    """Ideal generated by gens: the Z-span of g and g*sqrt(-5)."""
    return hnf([g for g in gens] + [mul(g, (0, 1)) for g in gens])

def imul(I, J):
    return ideal(*[mul(u, v) for u in I for v in J])

def N(I): return I[0][0] * I[1][1]

p  = ideal((2, 0), (1, 1));   q = ideal((3, 0), (1, 1));   qb = ideal((3, 0), (1, -1))
print("p =", p, " N(p) =", N(p), "   q =", q, " N(q) =", N(q), "   qbar =", qb)
print("p^2       == (2):        ", imul(p, p) == ideal((2, 0)))
print("q qbar    == (3):        ", imul(q, qb) == ideal((3, 0)))
print("p q       == (1+sqrt-5): ", imul(p, q) == ideal((1, 1)))
print("p qbar    == (1-sqrt-5): ", imul(p, qb) == ideal((1, -1)))

In [ ]:
def contains(I, u):
    (a, b), (_, d) = I
    k, r = divmod(u[0], a)
    return r == 0 and (u[1] - k*b) % d == 0

def principal_generator(I):
    """Search for an element of norm N(I) in I (norm a^2+5b^2 = N(I) bounds the search)."""
    n = N(I)
    for a in range(-isqrt(n), isqrt(n) + 1):
        for b in range(-isqrt(n // 5), isqrt(n // 5) + 1):
            if norm((a, b)) == n and contains(I, (a, b)):
                return (a, b)
    return None

for name, I in [("p", p), ("q", q), ("p q", imul(p, q)), ("q^2", imul(q, q)), ("p q^2", imul(p, imul(q, q)))]:
    print(f"{name:6s} N = {N(I):3d}  generator: {principal_generator(I)}")

$\mathfrak p$ and $\mathfrak q$ are not principal, but $\mathfrak p\mathfrak q=(1+\sqrt{-5})$ is: the classes of $\mathfrak p$ and $\mathfrak q$ are equal and of order $2$. $\mathfrak q^2$, of norm $9$, is generated by $2-\sqrt{-5}$ (or its negative), while $\mathfrak p\mathfrak q^2$, of norm $18$, is again not principal: the class group $\{1,[\mathfrak p]\}$ has order $2$, and an ideal is principal exactly when it contains an even number of non-principal prime factors, counted with multiplicity.

In [ ]:
def class_number(D):
    """Number of reduced primitive forms of discriminant D < 0."""
    h = 0
    a = 1
    while 3*a*a <= -D:
        for b in range(-a + 1, a + 1):
            if (b*b - D) % (4*a): continue
            c = (b*b - D) // (4*a)
            if c < a or gcd(gcd(a, abs(b)), c) != 1: continue
            if b < 0 and a == c: continue
            h += 1
        a += 1
    return h

def is_fundamental(D):
    if D % 4 == 1:
        m = -D
        return all(m % (k*k) for k in range(3, isqrt(m) + 1, 2)) and m % 4 != 0
    if D % 4 == 0:
        m = D // 4
        if m % 4 not in (2, 3): return False
        return all(m % (k*k) for k in range(2, isqrt(-m) + 1))
    return False

print("h(-20) =", class_number(-20), "  h(-4) =", class_number(-4), "  h(-23) =", class_number(-23), "  h(-47) =", class_number(-47))
print("class number one, -2000 < D < 0:", [D for D in range(-1, -2000, -1) if is_fundamental(D) and class_number(D) == 1])

Exactly nine discriminants, $-3,-4,-7,-8,-11,-19,-43,-67,-163$, and the search can be pushed much further without finding a tenth: Gauss conjectured that the list is complete, and Kurt Heegner (1952), then Alan Baker and Harold Stark (1966–67), proved it. The other values show that class groups grow: $h(-23)=3$, $h(-47)=5$, and $h(D)$ is roughly of size $\sqrt{|D|}$ (Siegel).

### P5.2

In [ ]:
import sympy as sp, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
x = sp.Symbol("x")
types = {}
for p in sp.primerange(2, 200):
    _, facs = sp.factor_list(x**3 - 2, modulus=p)
    ef = sorted(((e, sp.degree(g, x)) for g, e in facs), reverse=True)
    assert sum(e*f for e, f in ef) == 3
    types.setdefault(tuple(ef), []).append(p)
for t, ps in types.items():
    print("(e, f) pairs", t, ":", ps if len(ps) < 12 else ps[:12] + ["..."], f"({len(ps)} primes)")

Every prime satisfies $\sum e_if_i=3$. Only $2$ and $3$ ramify ($x^3-2\equiv x^3\pmod 2$ and $\equiv(x+1)^3\pmod3$); they are the primes dividing the discriminant $-108$. For $p\equiv2\pmod3$, cubing is a bijection of $\mathbb F_p$, so $x^3-2$ has exactly one root and the type is $(1,1),(1,2)$: about half of all primes. For $p\equiv1\pmod3$ there are three roots or none, and "none" (an inert prime, $f=3$) happens about twice as often as complete splitting. Chebotarev's theorem (Notebook 4) predicts the frequencies $1/2$, $1/3$, $1/6$ from the cycle types of $\mathfrak S_3$; here $23/44$, $16/44$ and $5/44$ of the unramified primes below $200$.

### P5.3

In [ ]:
import random, time

def negacyclic_schoolbook(a, b, q):
    n = len(a); c = [0]*n
    for i in range(n):
        ai = a[i]
        if ai == 0: continue
        for j in range(n):
            k = i + j
            if k < n: c[k] = (c[k] + ai*b[j]) % q
            else:     c[k - n] = (c[k - n] - ai*b[j]) % q     # x^n = -1
    return c

def find_psi(n, q):
    """A primitive 2n-th root of unity mod q (needs q = 1 mod 2n)."""
    for g in range(2, q):
        psi = pow(g, (q - 1) // (2*n), q)
        if pow(psi, n, q) == q - 1:
            return psi

def ntt(a, w, q):
    """Cyclic DFT over Z_q, recursive radix-2: values a(w^k), k = 0..n-1."""
    n = len(a)
    if n == 1: return a[:]
    even = ntt(a[0::2], w*w % q, q); odd = ntt(a[1::2], w*w % q, q)
    out = [0]*n; t = 1
    for k in range(n // 2):
        u, v = even[k], t*odd[k] % q
        out[k], out[k + n//2] = (u + v) % q, (u - v) % q
        t = t*w % q
    return out

def negacyclic_ntt(a, b, q, psi):
    """Evaluate at the roots psi^(2k+1) of x^n+1 (twist by psi^i, then a cyclic DFT with w = psi^2), multiply pointwise, interpolate."""
    n = len(a); w = psi*psi % q
    A = ntt([a[i]*pow(psi, i, q) % q for i in range(n)], w, q)
    B = ntt([b[i]*pow(psi, i, q) % q for i in range(n)], w, q)
    C = [x*y % q for x, y in zip(A, B)]
    c = ntt(C, pow(w, q - 2, q), q)
    ninv, psiinv = pow(n, q - 2, q), pow(psi, q - 2, q)
    return [c[i]*ninv % q * pow(psiinv, i, q) % q for i in range(n)]

n, q = 512, 12289
print("q mod 2n =", q % (2*n))
psi = find_psi(n, q)
print("psi =", psi, " order 2n:", pow(psi, n, q) == q - 1)
random.seed(5)
a = [random.randrange(q) for _ in range(n)]; b = [random.randrange(q) for _ in range(n)]
t0 = time.perf_counter(); c1 = negacyclic_schoolbook(a, b, q); t1 = time.perf_counter()
c2 = negacyclic_ntt(a, b, q, psi); t2 = time.perf_counter()
print("NTT product equals schoolbook product:", c1 == c2)
print(f"schoolbook {1000*(t1-t0):.1f} ms,  NTT {1000*(t2-t1):.1f} ms")

The two products agree. The NTT is the Chinese remainder isomorphism $R_q\cong\prod_{k}\Z_q[x]/(x-\psi^{2k+1})$ made fast: evaluating at the $n$ roots of $x^n+1$ sends a product to a pointwise product, and the recursive structure of the roots of unity (a radix-2 FFT) does it in $O(n\log n)$ operations instead of $n^2$. Even in pure Python the gain is large; real implementations add precomputed tables and constant-time arithmetic.

In [ ]:
n, q = 256, 3329
print("q - 1 =", sp.factorint(q - 1), "  q mod 512 =", q % 512)
zeta = 17
print("order of 17 mod 3329:", sp.n_order(zeta, q))
# x^256 + 1 = prod over odd j < 256 of (x^2 - zeta^j): multiply the 128 quadratics and compare.
P = sp.Poly(1, x, modulus=q)
for j in range(1, 256, 2):
    P = P * sp.Poly(x**2 - pow(zeta, j, q), x, modulus=q)
print("product of the 128 quadratics == x^256 + 1 mod q:", P == sp.Poly(x**256 + 1, x, modulus=q))
print("does x^256+1 have a root mod q?", any(pow(r, 256, q) == q - 1 for r in range(q)))

Since $3328=2^8\cdot13$, $\mathbb F_{3329}^\times$ has elements of order $256$ but none of order $512$, so $x^{256}+1$ has no root; but $\zeta=17$ has order $256$, and the $128$ quadratics $x^2-\zeta^j$, $j$ odd, multiply to $x^{256}+1$. Each is irreducible ($\zeta^j$ is not a square, since a square root of it would have order $512$). So $3329$ splits in $\Z[\zeta_{512}]$ into $g=128$ primes with $e=1$ and $f=2$, and $efg=256$, as Theorem 5.14 demands. This is why Kyber's NTT stops one level early and ends with $128$ multiplications of degree-one polynomials modulo quadratics: the arithmetic of a Dedekind domain decides the shape of the algorithm.